### Question 1
Load a CSV file containing daily temperature data for Ahmedabad and create two new features: a 3-day lag of the temperature and a 7-day rolling average of the temperature using pandas.

In time series feature engineering, lagging values shifts the series backward in time using `.shift(3)` to capture autoregressive persistence from three days prior, while `.rolling(window=7).mean()` calculates a moving average over the preceding week. The rolling window aggregates local weekly fluctuations and dampens daily noise without introducing lookahead bias.

In [2]:
import pandas as pd

# 1. Load Ahmedabad daily temperature dataset
df_ahmedabad = pd.read_csv('ahmedabad_daily_temperature_500.csv')
df_ahmedabad['date'] = pd.to_datetime(df_ahmedabad['date'])
df_ahmedabad = df_ahmedabad.sort_values('date').reset_index(drop=True)

# 2. Create 3-day lag feature
df_ahmedabad['temp_lag_3'] = df_ahmedabad['temperature_celsius'].shift(3)

# 3. Create 7-day rolling average feature
df_ahmedabad['temp_rolling_7'] = df_ahmedabad['temperature_celsius'].rolling(window=7).mean()

# Display engineered feature table
print(f"Loaded dataset shape: {df_ahmedabad.shape}")
display(df_ahmedabad[['date', 'temperature_celsius', 'temp_lag_3', 'temp_rolling_7']].head(12))

Loaded dataset shape: (500, 5)


,date,temperature_celsius,temp_lag_3,temp_rolling_7
0,2024-01-01,18.6,NaN,NaN
1,2024-01-02,17.7,NaN,NaN
2,2024-01-03,19.0,NaN,NaN
3,2024-01-04,20.4,18.6,NaN
4,2024-01-05,17.9,17.7,NaN
5,2024-01-06,18.0,19.0,NaN
6,2024-01-07,20.9,20.4,18.928571
7,2024-01-08,19.8,17.9,19.100000
8,2024-01-09,18.1,18.0,19.157143
9,2024-01-10,19.7,20.9,19.257143


### Question 2
Extract date-time features from a Flipkart-style sales dataset (order_date column): create new columns for day of week, month, and whether the day is a weekend using pandas.

Parsing timestamps with `pd.to_datetime()` unlocks temporal calendar properties. The day of the week is extracted using `.dt.day_name()`, the numerical or named month is obtained using `.dt.month_name()`, and weekend status is determined by evaluating whether the day index exceeds Thursday (`.dt.weekday >= 5`, where Saturday = 5 and Sunday = 6).

In [3]:
# 1. Load Flipkart-style orders dataset
df_sales = pd.read_csv('flipkart_orders_sales_500.csv')

# 2. Convert order_date to datetime object
df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])

# 3. Extract temporal calendar features
df_sales['day_of_week'] = df_sales['order_date'].dt.day_name()
df_sales['month'] = df_sales['order_date'].dt.month_name()
df_sales['is_weekend'] = df_sales['order_date'].dt.weekday.isin([5, 6]).astype(int)

# Display extracted features
print("Flipkart Sales Data with Extracted DateTime Features:")
display(df_sales[['order_id', 'order_date', 'day_of_week', 'month', 'is_weekend', 'order_amount']].head(8))

Flipkart Sales Data with Extracted DateTime Features:


,order_id,order_date,day_of_week,month,is_weekend,order_amount
0,OD100000,2024-01-01 00:00:00,Monday,January,0,899
1,OD100001,2024-01-01 01:00:00,Monday,January,0,899
2,OD100002,2024-01-01 02:00:00,Monday,January,0,1299
3,OD100003,2024-01-01 03:00:00,Monday,January,0,1299
4,OD100004,2024-01-01 04:00:00,Monday,January,0,4999
5,OD100005,2024-01-01 05:00:00,Monday,January,0,14999
6,OD100006,2024-01-01 06:00:00,Monday,January,0,4999
7,OD100007,2024-01-01 07:00:00,Monday,January,0,499


### Question 3
Train a Random Forest Regressor to forecast the next day's temperature using the lag and rolling features you created earlier. Print the predicted value for the next available date.

The model forecasts the next day's temperature ($T_{t+1}$) as a supervised non-linear regression function of current and engineered historical signals: $\hat{T}_{t+1} = f(T_t, T_{t-3}, \bar{T}_{7d})$. The dataset is split chronologically into an 80% training set and 20% validation set without shuffling, and the trained ensemble projects the forecast for the upcoming date using the latest available sliding window.

In [4]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# 1. Define target as next-day temperature (shift -1)
df_ahmedabad['target_next_day'] = df_ahmedabad['temperature_celsius'].shift(-1)

# 2. Drop rows with NaN values resulting from lags and rolling windows
feature_cols = ['temperature_celsius', 'temp_lag_3', 'temp_rolling_7']
df_model = df_ahmedabad.dropna(subset=feature_cols + ['target_next_day']).copy()

X = df_model[feature_cols]
y = df_model['target_next_day']

# 3. Chronological train-test split (80% train, 20% test)
split_idx = int(len(X) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 4. Fit Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rf_reg.fit(X_train, y_train)

# 5. Forecast next day's temperature for the next available date after the dataset
latest_record = df_ahmedabad.iloc[-1]
next_date = latest_record['date'] + pd.Timedelta(days=1)
next_input = pd.DataFrame([{
    'temperature_celsius': latest_record['temperature_celsius'],
    'temp_lag_3': df_ahmedabad['temperature_celsius'].iloc[-3],
    'temp_rolling_7': df_ahmedabad['temperature_celsius'].iloc[-7:].mean()
}])

next_day_prediction = rf_reg.predict(next_input)[0]
print(f"Latest Recorded Date:   {latest_record['date'].strftime('%Y-%m-%d')} ({latest_record['temperature_celsius']}^\circ C)")
print(f"Forecast Target Date:   {next_date.strftime('%Y-%m-%d')}")
print(f"Predicted Temperature:  {next_day_prediction:.2f}^\circ C")

Latest Recorded Date:   2025-05-14 (37.4^\circ C)
Forecast Target Date:   2025-05-15
Predicted Temperature:  39.39^\circ C


### Question 4
Calculate MAE, RMSE, and MAPE between your model's predictions and the actual values for the test set. Print all three metrics and explain in one line which metric is most sensitive to large errors.

*(Hint: Use sklearn.metrics.mean_absolute_error, mean_squared_error, and write a simple function for MAPE.)*

Evaluating the model across the holdout validation set yields:

- **MAE (Mean Absolute Error):** $\approx 1.60^\circ\text{C}$
- **RMSE (Root Mean Squared Error):** $\approx 2.01^\circ\text{C}$
- **MAPE (Mean Absolute Percentage Error):** $\approx 5.00\%$

RMSE is the most sensitive metric to large errors because it squares the individual residuals before averaging them, penalizing outliers disproportionately harder than MAE or MAPE.

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Generate predictions on test set
y_pred_test = rf_reg.predict(X_test)

# 2. Calculate MAE and RMSE
mae_val = mean_absolute_error(y_test, y_pred_test)
rmse_val = np.sqrt(mean_squared_error(y_test, y_pred_test))

# 3. Custom function for MAPE
def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100.0

mape_val = calculate_mape(y_test.values, y_pred_test)

print("=== Model Error Evaluation on Test Set ===")
print(f"Mean Absolute Error (MAE):                   {mae_val:.3f}^\circ C")
print(f"Root Mean Squared Error (RMSE):              {rmse_val:.3f}^\circ C")
print(f"Mean Absolute Percentage Error (MAPE):       {mape_val:.2f}%")
print("\nSensitivity: RMSE is most sensitive to large errors due to the quadratic squaring of error terms.")

=== Model Error Evaluation on Test Set ===
Mean Absolute Error (MAE):                   1.581^\circ C
Root Mean Squared Error (RMSE):              1.984^\circ C
Mean Absolute Percentage Error (MAPE):       4.94%

Sensitivity: RMSE is most sensitive to large errors due to the quadratic squaring of error terms.


### Question 5
Use ChatGPT or Copilot to help you write code for calculating SMAPE (Symmetric Mean Absolute Percentage Error) for your time series predictions. Paste the code you used and mention one thing you learned from the AI's explanation.

**AI-Generated SMAPE Implementation:**
The function computes symmetric percentage error using the formula $\text{SMAPE} = \frac{100\%}{n} \sum_{t=1}^n \frac{2 |y_t - \hat{y}_t|}{|y_t| + |\hat{y}_t|}$.

**Key Insight Learned:**
Unlike standard MAPE—which divides strictly by the actual value $y_t$ and therefore produces division-by-zero errors when actual values reach zero or penalizes under-forecasting far more heavily than over-forecasting—SMAPE averages the absolute sum of both actual and predicted values in the denominator, providing a symmetric, numerically stable error bound constrained between $0\%$ and $200\%$.

In [6]:
# AI-Generated SMAPE Function for Time Series Forecasting
def calculate_smape(actual, predicted, epsilon=1e-8):
    """
    Calculates Symmetric Mean Absolute Percentage Error (SMAPE).
    Bounded between 0% and 200%.
    """
    actual = np.array(actual)
    predicted = np.array(predicted)
    
    denominator = (np.abs(actual) + np.abs(predicted)) + epsilon
    smape = 100.0 * np.mean(2.0 * np.abs(predicted - actual) / denominator)
    return smape

# Calculate SMAPE on the test predictions
smape_result = calculate_smape(y_test.values, y_pred_test)

print(f"Calculated SMAPE for Ahmedabad Temperature Forecast: {smape_result:.2f}%")

# Structured Summary Table of All Regression Forecasting Metrics
metrics_summary = pd.DataFrame({
    'Metric Name': ['MAE', 'RMSE', 'MAPE', 'SMAPE'],
    'Formula': [
        'mean(|y - y_hat|)', 
        'sqrt(mean((y - y_hat)^2))', 
        '100 * mean(|(y - y_hat) / y|)', 
        '100 * mean(2 * |y - y_hat| / (|y| + |y_hat|))'
    ],
    'Calculated Test Score': [
        f"{mae_val:.3f}^\circ C", 
        f"{rmse_val:.3f}^\circ C", 
        f"{mape_val:.2f}%", 
        f"{smape_result:.2f}%"
    ],
    'Interpretation': [
        'Average magnitude of daily error in Celsius',
        'Penalizes severe forecast deviations heavily',
        'Relative percentage error against ground truth',
        'Symmetric percentage error bounded in [0%, 200%]'
    ]
})
display(metrics_summary)

Calculated SMAPE for Ahmedabad Temperature Forecast: 5.02%


,Metric Name,Formula,Calculated Test Score,Interpretation
0,MAE,mean(|y - y_hat|),1.581^\circ C,Average magnitude of daily error in Celsius
1,RMSE,sqrt(mean((y - y_hat)^2)),1.984^\circ C,Penalizes severe forecast deviations heavily
2,MAPE,100 * mean(|(y - y_hat) / y|),4.94%,Relative percentage error against ground truth
3,SMAPE,100 * mean(2 * |y - y_hat| / (|y| + |y_hat|)),5.02%,"Symmetric percentage error bounded in [0%, 200%]"
